In [9]:
import torch
print("PyTorch GPU available:", torch.cuda.is_available())
print("GPU device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch GPU available: True
GPU device name: NVIDIA GeForce RTX 3050 Laptop GPU


In [10]:
# Imports
import sys
sys.path.append("../..")

import os
import torch
import torchaudio
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import Dataset, DatasetDict, concatenate_datasets, Audio
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)

from utils.audio import load_common_voice_dataset_from_tsv

In [16]:
# import pandas as pd

# # Example for English
# tsv_path = "../../full_datasets/asr/common_voice_13_0/audio/en/train.tsv"
# df = pd.read_csv(tsv_path, sep="\t")
# print(df.columns)

# tsv_path = "../../full_datasets/asr/common_voice_13_0/audio/hi/train.tsv"
# df = pd.read_csv(tsv_path, sep="\t")
# print(df.columns)

Index(['version https://git-lfs.github.com/spec/v1'], dtype='object')
Index(['version https://git-lfs.github.com/spec/v1'], dtype='object')


In [12]:
# Config
MODEL_NAME = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000
DATA_BASE = "../../full_datasets/asr/common_voice_13_0/audio"
LANGS = ["en", "hi"]

In [13]:
# Load Processor
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

/home/hartz-byte/miniconda3/envs/polychat_gpu/lib/python3.10/site-packages/transformers/configuration_utils.py:309: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [14]:
# Load Datasets for en + hi
def load_lang_dataset(lang):
    base_path = os.path.join(DATA_BASE, lang)
    train = load_common_voice_dataset_from_tsv(os.path.join(base_path, "train.tsv"), base_path)
    val = load_common_voice_dataset_from_tsv(os.path.join(base_path, "dev.tsv"), base_path)
    test = load_common_voice_dataset_from_tsv(os.path.join(base_path, "test.tsv"), base_path)

    for ds in [train, val, test]:
        ds = ds.add_column("lang", [lang] * len(ds))

    train = train.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    val = val.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    test = test.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    return train, val, test

train_datasets, val_datasets, test_datasets = [], [], []

for lang in LANGS:
    tr, va, te = load_lang_dataset(lang)
    train_datasets.append(tr)
    val_datasets.append(va)
    test_datasets.append(te)

train_dataset = concatenate_datasets(train_datasets)
val_dataset = concatenate_datasets(val_datasets)
test_dataset = concatenate_datasets(test_datasets)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

print(f"Dataset loaded: {len(dataset['train'])} train, {len(dataset['validation'])} val, {len(dataset['test'])} test")

KeyError: "None of [Index(['path', 'sentence'], dtype='object')] are in the [columns]"

In [ ]:
# Preprocessing
def preprocess(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names)

In [ ]:
# Data Collator
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

In [ ]:
# Model
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    vocab_size=len(processor.tokenizer),
    pad_token_id=processor.tokenizer.pad_token_id,
)

In [ ]:
# Training
training_args = TrainingArguments(
    output_dir="../../models/asr-multilingual",
    group_by_length=True,
    per_device_train_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    learning_rate=3e-4,
    warmup_steps=500,
    logging_steps=10,
    save_total_limit=2,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)

In [ ]:
# Train
print("🔧 Starting multilingual ASR training...")
trainer.train()

In [ ]:
# Save
model.save_pretrained("../../models/asr-multilingual")
processor.save_pretrained("../../models/asr-multilingual")

print("Model and processor saved to ../../models/asr-multilingual")